In [72]:
# Install if needed: !pip install python-dotenv pandas requests

from dotenv import load_dotenv
load_dotenv(override=True)

import os
import base64
import requests
from pathlib import Path
import pandas as pd
import re
from IPython.display import display


In [73]:
# OpenRouter API setup
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY in your .env")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

model_name = "anthropic/claude-3.7-sonnet"

def encode_image(path: Path):
    """Convert image to OpenRouter format."""
    data = path.read_bytes()
    if data[:2] == b"\xff\xd8":
        media_type = "image/jpeg"
    elif data[:8] == b"\x89PNG\r\n\x1a\n":
        media_type = "image/png"
    else:
        media_type = "image/jpeg" if path.suffix.lower() in {".jpg", ".jpeg"} else "image/png"
    b64 = base64.b64encode(data).decode("utf-8")
    data_url = f"data:{media_type};base64,{b64}"
    return {"type": "image_url", "image_url": {"url": data_url}}

def call_openrouter(system_prompt, user_content, temperature=0):
    """Call OpenRouter API with system prompt and user content."""
    # OpenRouter format: system prompt goes in messages with role "system"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]
    
    payload = {
        "model": model_name,
        "temperature": temperature,
        "max_tokens": 2000,
        "messages": messages,
    }
    
    r = requests.post(OPENROUTER_URL, headers=OPENROUTER_HEADERS, json=payload, timeout=120)
    
    if r.status_code == 402:
        error_data = r.json() if r.text else {}
        error_msg = error_data.get("error", {}).get("message", r.text)
        raise RuntimeError(
            f"❌ Payment Required (402): {error_msg}\n"
            f"Fix: Increase API key limit at https://openrouter.ai/settings/keys"
        )
    
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


In [74]:
# Prompts
system_prompt = """
You are a micromobility parking enforcement officer.
Analyze this parking photo of a shared e-scooter or bicycle.
Rate it as PASS or FAIL and provide feedback accordingly.

Condense your feedback into a single, short sentence.
If FAIL, make it actionable so the rider knows what they need to improve.
"""

user_prompt = """
Analyze this parking photo according to the rules below and decide whether the vehicle is correctly parked. Each Bird or Spin vehicle must be evaluated individually.

Vehicle types to evaluate

Only evaluate these four vehicle types used in this market:
• Bird scooters (white-and-blue)
• Spin scooters (orange)
• Bird e-bikes (blue frame with black rear fender)
• Bird e-bikes (white frame with blue branding)

1. Identify the corral

A micromobility parking corral is defined only by one or more of the following indicators:
• A painted "P" symbol
• A painted scooter or bicycle icon
• Black-and-white posts or bollards arranged to outline a parking zone

Any of these alone is sufficient to confirm a corral.

2. Supportive indicators

These markings may help confirm corral boundaries when a primary indicator is present, but cannot define a corral by themselves:
• White painted boxes or boundary lines
• Painted rectangles or shapes
• Other random paint on the street or sidewalk

Supportive indicators only apply when they are white and align with known corral layouts.

3. Snow or partial visibility

• If a P symbol or scooter/bike icon is visible, treat that area as the corral even if boundary paint is partially missing or covered by snow.
• If a Bird or Spin vehicle is parked directly on top of, or centered over, a visible P symbol or scooter/bike icon, and is upright, treat it as correctly parked inside the corral (PASS) even if snow hides the rest of the markings.
• When black-and-white posts or bollards form a corral, only the area inside those posts counts as the corral. Any Bird or Spin vehicle outside the posts is outside the corral, even if it is close to a P symbol.
• Colored ground markings — including yellow, red, orange, blue, green, or construction/utility paint — are not necessarily micromobility corrals, and must not be interpreted as corrals even when they form a distinct box or shape.
• If no primary corral indicator (P symbol, scooter/bike icon, or black-and-white posts/bollards) is visible anywhere in the photo → FAIL.

4. Parking correctness

• The corral area is only the specific region defined by the corral indicators (around the P/icon or inside the posts/bollards). Do not assume that all nearby pavement is part of the corral. Any Bird or Spin vehicle outside that defined area → FAIL.
• Consider all Bird and Spin vehicles visible in the photo. If any of them is outside the corral → FAIL.
• If a vehicle is partly inside the corral, the majority of the vehicle must be inside to PASS.
• The vehicle must be upright, fully visible, and not held by a person.

Examples

• A scooter standing directly on the P symbol → PASS
• A scooter parked in the bike lane with no P symbol or posts visible → FAIL
• A scooter on the sidewalk while the corral with posts is behind it → FAIL
• Two scooters inside the corral → PASS
• Two scooters in the photo but one is outside the corral → FAIL
• A scooter partly inside the corral but the majority is inside → PASS
• Several scooters are clustered around the P symbol or inside the posts, but one Bird/Spin scooter is closer to the camera on unmarked pavement in front of the corral → FAIL (that front scooter is outside the corral).

"""


In [75]:
# Process all images in Denver nests photos folder
images_dir = Path("Denver nests photos")
image_paths = sorted([p for p in images_dir.glob("**/*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
print(f"Processing {len(image_paths)} images with temp=0.3...")

def extract_decision(response):
    """Extract PASS/FAIL decision from response."""
    response_upper = response.upper()
    if "PASS" in response_upper and "FAIL" not in response_upper:
        return "PASS"
    elif "FAIL" in response_upper:
        return "FAIL"
    else:
        return "UNKNOWN"

def extract_reason(response):
    """Extract reason (first sentence or whole response)."""
    return response.strip().split("\n")[0] if "\n" in response else response.strip()

results = []
for img_path in image_paths:
    # Extract expected result from filename
    name_lower = img_path.name.lower()
    expected = "PASS" if "pass" in name_lower else ("FAIL" if "fail" in name_lower else None)
    
    # Prepare user content (text + image)
    user_content = [
        {"type": "text", "text": user_prompt},
        encode_image(img_path),
    ]
    
    # Call API with temperature 0.3
    response = call_openrouter(system_prompt, user_content, temperature=0.3)
    decision = extract_decision(response)
    reason = extract_reason(response)
    
    results.append({
        "Image": img_path.name,
        "Expected": expected or "-",
        "Decision": decision,
        "Reason": reason,
    })
    print(f"✓ {img_path.name}: {decision}")

# Display results table
df = pd.DataFrame(results)
display(df)

# Save to CSV
df.to_csv("denver_results.csv", index=False)
print(f"\n✅ Saved {len(results)} results to denver_results.csv")


Processing 20 images with temp=0.3...
✓ denver_expected_fail_1.jpg: FAIL
✓ denver_expected_fail_10.jpg: FAIL
✓ denver_expected_fail_11.jpg: FAIL
✓ denver_expected_fail_12.jpg: FAIL
✓ denver_expected_fail_13.jpg: FAIL
✓ denver_expected_fail_14.jpg: PASS
✓ denver_expected_fail_2.jpg: FAIL
✓ denver_expected_fail_3.jpg: FAIL
✓ denver_expected_fail_4.jpg: FAIL
✓ denver_expected_fail_5.jpg: FAIL
✓ denver_expected_fail_6.jpg: FAIL
✓ denver_expected_fail_7.jpg: FAIL
✓ denver_expected_fail_8.jpg: FAIL
✓ denver_expected_fail_9.jpg: FAIL
✓ denver_expected_fail_result_fail_1.jpg: FAIL
✓ denver_expected_pass_1.jpg: PASS
✓ denver_expected_pass_3.jpg: PASS
✓ denver_expected_pass_4.jpg: PASS
✓ denver_expected_pass_5.jpg: PASS
✓ denver_expected_pass_6.jpg: PASS


,Image,Expected,Decision,Reason
0,denver_expected_fail_1.jpg,FAIL,FAIL,FAIL - The Bird scooter is parked on the sidew...
1,denver_expected_fail_10.jpg,FAIL,FAIL,"FAIL: No visible corral indicators (P symbol, ..."
2,denver_expected_fail_11.jpg,FAIL,FAIL,FAIL - Scooter is parked in snow outside the d...
3,denver_expected_fail_12.jpg,FAIL,FAIL,"FAIL: No visible corral indicators (P symbol, ..."
4,denver_expected_fail_13.jpg,FAIL,FAIL,FAIL - The Bird scooter is parked on a regular...
5,denver_expected_fail_14.jpg,FAIL,PASS,PASS - Scooters are properly parked in a desig...
6,denver_expected_fail_2.jpg,FAIL,FAIL,FAIL - The Bird scooter is parked outside the ...
7,denver_expected_fail_3.jpg,FAIL,FAIL,"FAIL - No visible corral indicators (P symbol,..."
8,denver_expected_fail_4.jpg,FAIL,FAIL,"FAIL - No visible P symbol, scooter/bike icon,..."
9,denver_expected_fail_5.jpg,FAIL,FAIL,"FAIL - No visible corral indicators (P symbol,..."



✅ Saved 20 results to denver_results.csv


In [76]:
# Summary statistics
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

# Filter out rows without expected labels
labeled_df = df[df["Expected"] != "-"].copy()

if len(labeled_df) == 0:
    print("⚠️ No images with expected labels (PASS/FAIL in filename)")
else:
    total = len(labeled_df)
    
    expected_pass_count = len(labeled_df[labeled_df["Expected"] == "PASS"])
    expected_fail_count = len(labeled_df[labeled_df["Expected"] == "FAIL"])
    
    # Calculate statistics
    correct_pass = len(labeled_df[(labeled_df["Expected"] == "PASS") & (labeled_df["Decision"] == "PASS")])
    incorrect_pass = len(labeled_df[(labeled_df["Expected"] == "PASS") & (labeled_df["Decision"] == "FAIL")])
    correct_fail = len(labeled_df[(labeled_df["Expected"] == "FAIL") & (labeled_df["Decision"] == "FAIL")])
    incorrect_fail = len(labeled_df[(labeled_df["Expected"] == "FAIL") & (labeled_df["Decision"] == "PASS")])
    
    summary_data = {
        "Metric": [
            "Total labeled images",
            "",
            "Expected PASS → PASS (correct)",
            "Expected PASS → FAIL (incorrect)",
            "",
            "Expected FAIL → FAIL (correct)",
            "Expected FAIL → PASS (incorrect)",
            "",
            "Overall accuracy",
        ],
        "Count": [
            total,
            "",
            correct_pass,
            incorrect_pass,
            "",
            correct_fail,
            incorrect_fail,
            "",
            correct_pass + correct_fail,
        ],
        "Percentage": [
            100.0,
            "",
            round(correct_pass / expected_pass_count * 100, 1) if expected_pass_count > 0 else 0,
            round(incorrect_pass / expected_pass_count * 100, 1) if expected_pass_count > 0 else 0,
            "",
            round(correct_fail / expected_fail_count * 100, 1) if expected_fail_count > 0 else 0,
            round(incorrect_fail / expected_fail_count * 100, 1) if expected_fail_count > 0 else 0,
            "",
            round((correct_pass + correct_fail) / total * 100, 1) if total > 0 else 0,
        ],
    }
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)
    
    print(f"\n📊 Breakdown:")
    print(f"   Expected PASS images: {expected_pass_count}")
    print(f"   Expected FAIL images: {expected_fail_count}")


SUMMARY STATISTICS


,Metric,Count,Percentage
0,Total labeled images,20,100.0
1,,,
2,Expected PASS → PASS (correct),5,100.0
3,Expected PASS → FAIL (incorrect),0,0.0
4,,,
5,Expected FAIL → FAIL (correct),14,93.3
6,Expected FAIL → PASS (incorrect),1,6.7
7,,,
8,Overall accuracy,19,95.0



📊 Breakdown:
   Expected PASS images: 5
   Expected FAIL images: 15


In [77]:
# Test OpenRouter API connection
import requests
import os

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    print("❌ No OPENROUTER_API_KEY found in .env")
else:
    print(f"✅ API Key found: {OPENROUTER_API_KEY[:10]}...")
    
    # Test with models endpoint
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    try:
        r = requests.get("https://openrouter.ai/api/v1/models", headers=headers, timeout=10)
        if r.status_code == 200:
            print("✅ OpenRouter connection successful!")
            data = r.json()
            models = [m["id"] for m in data.get("data", [])]
            claude_models = [m for m in models if "claude" in m.lower() and "3.7" in m]
            if claude_models:
                print(f"\n✅ Claude 3.7 models available:")
                for m in claude_models[:5]:  # Show first 5
                    print(f"   - {m}")
            else:
                print("\n⚠️ No Claude 3.7 models found (but connection works)")
        elif r.status_code == 401:
            print("❌ Unauthorized (401) - API key is invalid")
        elif r.status_code == 402:
            print("❌ Payment Required (402) - Check your account credits/limits")
        else:
            print(f"⚠️ Unexpected status {r.status_code}: {r.text[:200]}")
    except Exception as e:
        print(f"❌ Error: {e}")


✅ API Key found: sk-or-v1-3...
✅ OpenRouter connection successful!

✅ Claude 3.7 models available:
   - anthropic/claude-3.7-sonnet:thinking
   - anthropic/claude-3.7-sonnet
